In [1]:
"""
Step 2: Generate synthetic irrelevant papers.

Input:  a CSV of your REAL irrelevant papers (title, abstract, doi/wos_id, and a
        short trap_type note you assign by hand, e.g. "animal_source",
        "wrong_property_focus", "insect_source")
Output: a CSV of DRAFT synthetic irrelevant papers for you to read and edit
        before they go anywhere near training data.

This script does NOT call an LLM automatically. It builds the prompts for you,
one per trap category, so you control quality batch by batch instead of
generating 140 at once. Copy the printed prompts into Claude/GPT/Gemini
yourself, batch by batch, and paste results into the output CSV.
"""

import pandas as pd

# ---- 1. Load your real irrelevant papers (fill in labels from step1 first) ----
real = pd.read_csv("step1_labeling_sheet.csv")
real_irrelevant = real[real["label"].str.lower() == "irrelevant"].copy()
print(f"Found {len(real_irrelevant)} real irrelevant papers.")

# ---- 2. Assign a trap_type to each by hand (edit this mapping) ----
# Do this once, manually, by skimming the 30 titles/abstracts.
# Categories should match your actual exclusion list:
#   animal_source, insect_source, microbial_source, seafood_source,
#   poultry_source, fish_source, wrong_property_focus (e.g. nutrition only,
#   no functional property), review_type
TRAP_CATEGORIES = [
    "animal_source",
    "insect_source",
    "microbial_source",
    "fish_or_seafood_source",
    "poultry_source",
    "wrong_property_focus",
    "non_protein_material",
    "fungal_source",
]

# ---- 3. Build a batch prompt per trap category ----
# Generate ~15-20 per category rather than 140 in one call, so you can
# review quality between batches.
PROMPT_TEMPLATE = """You are helping build a training dataset of REALISTIC but
IRRELEVANT paper titles and abstracts for a plant-protein functional-properties
classifier. The papers must be irrelevant for this specific reason: {trap_type}.

Rules:
- Write {n} unique synthetic paper titles + abstracts.
- Style and abstract structure must closely match real food-science journal abstracts
  (methods, results, numbers, conclusion).
- Each one should be a NEAR MISS: it should contain some surface-level keyword
  overlap with plant protein functional-property research (e.g. mentions
  "solubility", "gelation", "protein isolate", "modification"), so it is a
  believable classification trap, not an obviously unrelated paper.
- Do NOT reuse real paper titles. Do NOT plagiarize existing abstracts.
- Vary the specific plant/animal/fish/seafood/fungal/non-protein-material/insect/microbial source, the specific
  functional property studied, and the specific processing/modification method
  across the {n} examples so they are not repetitive.

Here are 2-3 real examples of this trap type from my dataset, for style reference only,
do not copy them directly:
{seed_examples}

Output as a CSV with exactly these columns: title, abstract, trap_type
"""

def build_prompt(trap_type: str, seed_examples: list[str], n: int = 15) -> str:
    seeds_text = "\n\n".join(seed_examples[:3])
    return PROMPT_TEMPLATE.format(trap_type=trap_type, n=n, seed_examples=seeds_text)


if __name__ == "__main__":
    # Example: print one batch prompt at a time. Fill trap_type column in
    # step1 output by hand first, then this will group seeds automatically.
    if "trap_type" not in real_irrelevant.columns:
        print("\nNOTE: add a 'trap_type' column to your labeled irrelevant papers")
        print("(one of:", TRAP_CATEGORIES, ") before running this for real.\n")
        print("Showing a generic example prompt instead:\n")
        example_seeds = [
            "Title: Synergistic effects of glycosylation and ginger essential oils on "
            "soy protein isolate- Artemisia sphaerocephala Krasch. gum composite films "
            "for chilled grass carp preservation\nAbstract: (fish preservation film study)",
            "Title: Mild high hydrostatic pressure processing: Effects on techno-functional "
            "properties and allergenicity of ovalbumin\nAbstract: (egg protein functionality study)",
        ]
        print(build_prompt("fish_or_seafood_source", example_seeds, n=15))
    else:
        for trap in TRAP_CATEGORIES:
            seeds = real_irrelevant[real_irrelevant["trap_type"] == trap]
            if len(seeds) == 0:
                continue
            seed_texts = [
                f"Title: {r.title}\nAbstract: {r.abstract[:300]}..."
                for r in seeds.itertuples()
            ]
            print(f"\n{'='*80}\nBATCH PROMPT FOR: {trap}\n{'='*80}\n")
            print(build_prompt(trap, seed_texts, n=15))

Found 46 real irrelevant papers.

BATCH PROMPT FOR: animal_source

You are helping build a training dataset of REALISTIC but
IRRELEVANT paper titles and abstracts for a plant-protein functional-properties
classifier. The papers must be irrelevant for this specific reason: animal_source.

Rules:
- Write 15 unique synthetic paper titles + abstracts.
- Style and abstract structure must closely match real food-science journal abstracts
  (methods, results, numbers, conclusion).
- Each one should be a NEAR MISS: it should contain some surface-level keyword
  overlap with plant protein functional-property research (e.g. mentions
  "solubility", "gelation", "protein isolate", "modification"), so it is a
  believable classification trap, not an obviously unrelated paper.
- Do NOT reuse real paper titles. Do NOT plagiarize existing abstracts.
- Vary the specific plant/animal/fish/seafood/fungal/non-protein-material/insect/microbial source, the specific
  functional property studied, and the spe